# Data Cleaning & Feature Engineering Pipeline

## Overview
This notebook demonstrates the end-to-end  data cleaning, description preprocessing, granular skill extraction, skill matrix creation, and feature engineering pipeline for the **Career Intelligence Engine**.

### 1. Module Imports & Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root directory to sys.path
sys.path.append(os.path.join("..", ".."))

from ml.src.data.load_data import load_raw_postings, get_processed_data_dir
from ml.src.data.clean_data import clean_postings_dataframe, filter_cs_tech_jobs
from ml.src.features.skill_dictionary import extract_skills_from_text, SKILL_CATALOG
from ml.src.features.feature_engineering import generate_skill_tables

### 2. Loading Processed Datasets

In [ ]:
processed_dir = os.path.join("..", "data", "processed")
cs_postings_path = os.path.join(processed_dir, "cs_job_postings.csv")
matrix_path = os.path.join(processed_dir, "job_skill_matrix.csv")
long_skills_path = os.path.join(processed_dir, "job_skills_long.csv")

cs_df = pd.read_csv(cs_postings_path)
skill_matrix = pd.read_csv(matrix_path)
skills_long = pd.read_csv(long_skills_path)

print(f"CS Postings Shape: {cs_df.shape}")
print(f"Skill Matrix Shape: {skill_matrix.shape}")
print(f"Long Skill Rows: {len(skills_long)}")

### 3. Inspection of Cleaned Job Descriptions
Demonstrating raw HTML description vs preprocessed `description_clean`.

In [ ]:
sample_row = cs_df.iloc[0]
print("=== RAW DESCRIPTION SAMPLE (First 250 chars) ===")
print(repr(str(sample_row['description'])[:250]))
print("\n=== CLEANED DESCRIPTION SAMPLE (First 250 chars) ===")
print(repr(str(sample_row['description_clean'])[:250]))

### 4. Binary Skill Matrix & Micro-Skill Extraction Inspection

In [ ]:
print("Sample Binary Skill Matrix Rows:")
skill_matrix.head(5)

### 5. Engineered Features Analysis

In [ ]:
feature_cols = ['job_id', 'title', 'role_category', 'skill_count', 'skill_density', 'experience_level_encoded', 'is_remote', 'has_salary']
cs_df[feature_cols].head(10)

### 6. Top Technical Skills Demanded in CS/Technology Roles

In [ ]:
top_20 = skills_long['skill'].value_counts().head(20)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_20.values, y=top_20.index, palette="viridis")
plt.title("Top 20 Most Demanded Technical Skills in CS/Technology Job Postings")
plt.xlabel("Posting Count")
plt.ylabel("Skill")
plt.tight_layout()
plt.show()

### 7. Skills Demanded by Role Category

In [ ]:
merged_skills = skills_long.merge(cs_df[['job_id', 'role_category']], on='job_id', how='left')
role_skill_counts = merged_skills.groupby(['role_category', 'skill']).size().reset_index(name='count')
top_by_role = role_skill_counts.sort_values(['role_category', 'count'], ascending=[True, False]).groupby('role_category').head(5)
print(top_by_role.to_string(index=False))